In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("/Users/samuelgirard/work/geometric_flow/pix_mapping/pix_irt_outcome.csv")

count    100000.000000
mean         31.871250
std           3.079306
min           1.000000
25%          31.000000
50%          32.000000
75%          32.000000
max         159.000000
Name: count, dtype: float64

In [20]:
import numpy as np
import pandas as pd

def dataframe_to_seqs(
    df,
    user_col="user",
    item_col="item",
    resp_col="correct",
    order_cols=None,     # e.g. ["timestamp"] if available
    threshold=0.5,       # for float outcomes
    min_user_len=5
):
    cols = [user_col, item_col, resp_col] + (order_cols or [])
    x = df[cols].dropna(subset=[user_col, item_col, resp_col]).copy()

    # Keep temporal order inside each user
    if order_cols:
        x = x.sort_values([user_col] + order_cols, kind="mergesort")
    else:
        x = x.reset_index(drop=False).sort_values([user_col, "index"], kind="mergesort")

    # Reindex user/item to 0..N-1
    u_idx, u_vals = pd.factorize(x[user_col], sort=False)
    i_idx, i_vals = pd.factorize(x[item_col], sort=False)

    r = x[resp_col].to_numpy()
    if np.issubdtype(r.dtype, np.floating):
        r = (r >= threshold).astype(np.int64)
    else:
        r = r.astype(np.int64)

    x["_u"] = u_idx.astype(np.int64)
    x["_i"] = i_idx.astype(np.int64)
    x["_r"] = r

    seqs = []
    for _, g in x.groupby("_u", sort=False):
        seq = list(zip(g["_i"].tolist(), g["_r"].tolist()))
        if len(seq) >= min_user_len:
            seqs.append(seq)

    return {
        "seqs": seqs,
        "n_users": len(seqs),
        "n_items": len(i_vals),
        "user_values": u_vals,   # local user index -> original user id
        "item_values": i_vals,   # local item index -> original item id
    }

def split_user_sequences(seqs, train_frac=0.8, min_test_len=1):
    train_seqs, test_seqs = [], []
    for seq in seqs:
        cut = int(len(seq) * train_frac)
        if len(seq) - cut >= min_test_len and cut > 0:
            train_seqs.append(seq[:cut])
            test_seqs.append(seq[cut:])
    return train_seqs, test_seqs

In [24]:
# df already loaded from pix_irt_outcome.csv
prep = dataframe_to_seqs(df, user_col="user", item_col="item", resp_col="correct")
seqs = prep["seqs"]
n_items = prep["n_items"]

train_seqs, test_seqs = split_user_sequences(seqs, train_frac=0.8)

W_hat, z0_hat = fit_item_embeddings(
    train_seqs, n_items=n_items, d=3,
    kappa=5.0, lam=1.0, eta_pos=0.5, eta_neg=0.5,
    use_prox=True, learn_user_init=True,
    epochs=3, lr=1e-1, device="cpu", seed=0
)

test_bce = heldout_bce(
    test_seqs, W_hat, z0_hat,
    kappa=5.0, lam=1.0, eta_pos=0.5, eta_neg=0.5, use_prox=True
)
print("heldout BCE:", test_bce)

epoch 001 | mean loss/user = 26.3189


KeyboardInterrupt: 

In [ ]:
def binary_auc_from_scores(y_true, y_score):
    """Rank-based AUC (Mann–Whitney U). Returns np.nan if only one class."""
    y_true = np.asarray(y_true, dtype=np.int64)
    y_score = np.asarray(y_score, dtype=np.float64)

    n_pos = int((y_true == 1).sum())
    n_neg = int((y_true == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    order = np.argsort(y_score, kind="mergesort")
    ranks = np.empty_like(order, dtype=np.float64)

    i = 0
    rank = 1.0
    while i < len(order):
        j = i + 1
        while j < len(order) and y_score[order[j]] == y_score[order[i]]:
            j += 1
        avg_rank = (rank + (rank + (j - i) - 1.0)) / 2.0
        ranks[order[i:j]] = avg_rank
        rank += (j - i)
        i = j

    sum_ranks_pos = ranks[y_true == 1].sum()
    auc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc)


def evaluate_sequences_metrics(
    seqs, W, z0,
    *,
    kappa=5.0, lam=1.0, eta_pos=0.5, eta_neg=1.0,
    use_prox=True,
    decision_threshold=0.5,
):
    """Teacher-forced evaluation over sequences: BCE, AUC, accuracy."""
    W = np_row_normalize(W)
    z0 = np_row_normalize(z0)

    total_nll = 0.0
    total_T = 0
    y_true, y_prob = [], []

    for u, seq in enumerate(seqs):
        z = z0[u].copy()
        for (i, r) in seq:
            w = W[i]
            s = float(kappa * (z @ w))
            p = float(sigmoid_np(s))

            y = int(r)
            y_true.append(y)
            y_prob.append(p)

            total_nll += -(y * np.log(max(p, 1e-12)) + (1 - y) * np.log(max(1 - p, 1e-12)))
            total_T += 1

            if y == 1:
                Pi = w - float(z @ w) * z
                z = z + eta_pos * kappa * (1.0 - p) * Pi
            else:
                proj = float(z @ w) * w
                gamma = eta_neg * p
                shrink = (gamma * lam) / (1.0 + gamma * lam) if use_prox else (gamma * lam)
                z = z - shrink * proj

            z = np_normalize(z)

    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    y_pred = (y_prob >= decision_threshold).astype(np.int64)

    bce = float(total_nll / max(total_T, 1))
    acc = float((y_pred == y_true).mean()) if total_T > 0 else float("nan")
    auc = binary_auc_from_scores(y_true, y_prob)

    return {
        "bce": bce,
        "auc": auc,
        "accuracy": acc,
        "n_obs": int(total_T),
    }


metrics_test = evaluate_sequences_metrics(
    test_seqs,
    W_hat,
    z0_hat,
    kappa=5.0,
    lam=1.0,
    eta_pos=0.5,
    eta_neg=1.0,
    use_prox=True,
    decision_threshold=0.5,
)
metrics_test

In [19]:

# 1) Create TRUE item embeddings W_true on the sphere + TRUE user initial states z0_true
# 2) Generate interaction sequences (item_id, response) from your model
# 3) Learn item embeddings W_hat (and optionally user z0_hat) by backprop through your geometric flow
# 4) Evaluate recovery metrics vs W_true (rotation-aware + rotation-invariant)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# =========================
# NumPy helpers
# =========================
def np_row_normalize(X, eps=1e-12):
    X = np.asarray(X, float)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(n, eps)

def np_normalize(v, eps=1e-12):
    v = np.asarray(v, float).reshape(-1)
    n = np.linalg.norm(v)
    return v / max(n, eps)

def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def stable_softmax_np(logits):
    x = logits - np.max(logits)
    p = np.exp(x)
    return p / p.sum()

# =========================
# SIMULATOR: generate sequences from TRUE embeddings
# =========================
def simulate_sequences_from_true(
    W_true, z0_true, horizon,
    *,
    kappa=5.0, lam=1.0, eta_pos=0.5, eta_neg=1.0,
    use_prox=True,
    seed=0
):
    """
    W_true: (n_items,d) unit rows
    z0_true: (n_users,d) unit rows
    returns: seqs = list of user sequences, each [(item_id, r), ...]
    """
    rng = np.random.default_rng(seed)
    W = np_row_normalize(W_true)
    Z0 = np_row_normalize(z0_true)

    n_users, d = Z0.shape
    n_items = W.shape[0]

    seqs = []
    for u in range(n_users):
        z = Z0[u].copy()
        seq = []
        for t in range(horizon):
            # Exposure policy: pi(i|z) ∝ exp(kappa z^T w_i)
            logits = kappa * (W @ z)
            pi = stable_softmax_np(logits)
            i = int(rng.choice(n_items, p=pi))
            w = W[i]

            # Feedback: r ~ Bernoulli(sigmoid(kappa z^T w))
            s = float(kappa * (z @ w))
            p = float(sigmoid_np(s))
            r = int(rng.random() < p)

            # Update z (your geometric flow)
            if r == 1:
                Pi = w - float(z @ w) * z
                z = z + eta_pos * kappa * (1.0 - p) * Pi
            else:
                # Proj_w(z) for unit w: (z^T w) w
                proj = float(z @ w) * w
                gamma = eta_neg * p
                if use_prox:
                    shrink = (gamma * lam) / (1.0 + gamma * lam)
                else:
                    shrink = gamma * lam
                z = z - shrink * proj

            z = np_normalize(z)
            seq.append((i, r))
        seqs.append(seq)
    return seqs

# =========================
# PyTorch helpers
# =========================
def torch_row_normalize(X, eps=1e-12):
    return X / X.norm(dim=-1, keepdim=True).clamp_min(eps)

def torch_vec_normalize(x, eps=1e-12):
    return x / x.norm(dim=-1, keepdim=True).clamp_min(eps)

# =========================
# LEARNER: learn W from sequences by unrolling your flow
# =========================
class GeoFlowEmbeddingLearner(nn.Module):
    """
    Learns item embeddings W (n_items,d) from sequences (i_t, r_t),
    with deterministic hidden state evolution z_{t+1}=f(z_t, w_{i_t}, r_t).

    Observation model:
      p_t = sigmoid(kappa * z_t^T w_{i_t})
      loss uses BCEWithLogits with logit = kappa * z_t^T w_{i_t}
    """
    def __init__(self, n_items, d, n_users, device="cpu"):
        super().__init__()
        self.W_raw = nn.Parameter(torch.randn(n_items, d, device=device))
        self.z0_raw = nn.Parameter(torch.randn(n_users, d, device=device))
        self.device = device

    def forward_sequence(
        self, user_id, item_ids, responses,
        *,
        kappa=5.0, lam=1.0, eta_pos=0.5, eta_neg=1.0,
        use_prox=True,
        eps=1e-12
    ):
        # constrain to sphere
        W = torch_row_normalize(self.W_raw, eps)                # (n_items,d)
        z = torch_vec_normalize(self.z0_raw[user_id], eps)      # (d,)

        total_loss = torch.tensor(0.0, device=self.device)

        for t in range(item_ids.shape[0]):
            i = item_ids[t]
            r = responses[t]        # scalar float 0/1

            w = W[i]
            dot = torch.dot(z, w)
            logit = kappa * dot
            p = torch.sigmoid(logit)

            # BCE on r
            total_loss = total_loss + F.binary_cross_entropy_with_logits(logit, r)

            # state update (teacher-forced with observed r)
            if r.item() == 1.0:
                Pi = w - dot * z
                z = z + eta_pos * kappa * (1.0 - p) * Pi
            else:
                proj = dot * w                     # Proj_w(z) for unit w
                gamma = eta_neg * p
                shrink = (gamma * lam) / (1.0 + gamma * lam) if use_prox else (gamma * lam)
                z = z - shrink * proj

            z = torch_vec_normalize(z, eps)

        return total_loss

    @torch.no_grad()
    def get_W(self):
        return torch_row_normalize(self.W_raw).cpu().numpy()

    @torch.no_grad()
    def get_z0(self):
        return torch_row_normalize(self.z0_raw).cpu().numpy()

def fit_item_embeddings(
    seqs, n_items, d,
    *,
    kappa=5.0, lam=1.0, eta_pos=0.5, eta_neg=1.0,
    use_prox=True,
    learn_user_init=True,
    epochs=30,
    lr=1e-2,
    device="cpu",
    seed=0
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    n_users = len(seqs)
    model = GeoFlowEmbeddingLearner(n_items, d, n_users, device=device).to(device)

    if not learn_user_init:
        model.z0_raw.requires_grad_(False)

    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # Preconvert sequences to tensors
    seq_tensors = []
    for u, seq in enumerate(seqs):
        item_ids = torch.tensor([it for (it, r) in seq], dtype=torch.long, device=device)
        responses = torch.tensor([float(r) for (it, r) in seq], dtype=torch.float32, device=device)
        seq_tensors.append((u, item_ids, responses))

    for ep in range(1, epochs + 1):
        total = 0.0
        perm = np.random.permutation(len(seq_tensors))
        for idx in perm:
            u, item_ids, responses = seq_tensors[idx]

            opt.zero_grad()
            loss = model.forward_sequence(
                u, item_ids, responses,
                kappa=kappa, lam=lam, eta_pos=eta_pos, eta_neg=eta_neg,
                use_prox=use_prox
            )
            loss.backward()
            opt.step()

            # keep parameters near sphere (optional but usually stabilizes)
            with torch.no_grad():
                model.W_raw.copy_(torch_row_normalize(model.W_raw))
                if learn_user_init:
                    model.z0_raw.copy_(torch_row_normalize(model.z0_raw))

            total += float(loss.item())

        print(f"epoch {ep:03d} | mean loss/user = {total/len(seq_tensors):.4f}")

    return model.get_W(), model.get_z0()

# =========================
# METRICS: compare W_hat vs W_true
# =========================
def orthogonal_procrustes_align(W_hat, W_true):
    """
    Find orthogonal R minimizing ||W_hat R - W_true||_F.
    """
    A = W_hat.T @ W_true
    U, _, Vt = np.linalg.svd(A)
    R = U @ Vt
    return W_hat @ R, R

def embedding_metrics(W_true, W_hat, topk=5):
    Wt = np_row_normalize(W_true)
    Wh = np_row_normalize(W_hat)

    # Rotation-aware: align by Procrustes then compare item-by-item
    Wh_aligned, _ = orthogonal_procrustes_align(Wh, Wt)

    cos_i = np.sum(Wh_aligned * Wt, axis=1)
    cos_i = np.clip(cos_i, -1.0, 1.0)
    ang_deg = np.degrees(np.arccos(cos_i))
    rel_frob = np.linalg.norm(Wh_aligned - Wt, "fro") / np.linalg.norm(Wt, "fro")

    # Rotation-invariant: compare pairwise cosine matrices
    C_true = Wt @ Wt.T
    C_hat  = Wh @ Wh.T
    iu = np.triu_indices(C_true.shape[0], k=1)
    ct, ch = C_true[iu], C_hat[iu]
    pair_corr = float(np.corrcoef(ct, ch)[0, 1])
    pair_mse = float(np.mean((ct - ch) ** 2))

    # Neighbor overlap@k (rotation-invariant)
    def topk_neighbors(C, k):
        C2 = C.copy()
        np.fill_diagonal(C2, -np.inf)
        return np.argsort(-C2, axis=1)[:, :k]

    nn_true = topk_neighbors(C_true, topk)
    nn_hat  = topk_neighbors(C_hat,  topk)
    overlap = []
    for i in range(Wt.shape[0]):
        overlap.append(len(set(nn_true[i]) & set(nn_hat[i])) / topk)

    return {
        "mean_cos_aligned": float(np.mean(cos_i)),
        "median_cos_aligned": float(np.median(cos_i)),
        "mean_angle_deg_aligned": float(np.mean(ang_deg)),
        "median_angle_deg_aligned": float(np.median(ang_deg)),
        "rel_frobenius_after_align": float(rel_frob),
        "pairwise_cos_corr": pair_corr,
        "pairwise_cos_mse": pair_mse,
        f"nn_overlap@{topk}": float(np.mean(overlap)),
    }

# Optional: held-out predictive BCE (teacher-forced)
def heldout_bce(seqs, W, z0, *, kappa=5.0, lam=1.0, eta_pos=0.5, eta_neg=1.0, use_prox=True):
    W = np_row_normalize(W)
    z0 = np_row_normalize(z0)

    total_loss = 0.0
    total_T = 0

    for u, seq in enumerate(seqs):
        z = z0[u].copy()
        for (i, r) in seq:
            w = W[i]
            s = float(kappa * (z @ w))
            p = float(sigmoid_np(s))

            total_loss += -(r * np.log(max(p, 1e-12)) + (1 - r) * np.log(max(1 - p, 1e-12)))
            total_T += 1

            if r == 1:
                Pi = w - float(z @ w) * z
                z = z + eta_pos * kappa * (1.0 - p) * Pi
            else:
                proj = float(z @ w) * w
                gamma = eta_neg * p
                shrink = (gamma * lam) / (1.0 + gamma * lam) if use_prox else (gamma * lam)
                z = z - shrink * proj
            z = np_normalize(z)

    return total_loss / max(total_T, 1)

# =========================
# RUN THE WHOLE PIPELINE
# =========================
def run_full_pipeline(
    *,
    n_items=60,
    n_users=40,
    d=8,
    horizon=200,
    kappa=5.0,
    lam=1.0,
    eta_pos=0.5,
    eta_neg=1.0,
    use_prox=True,
    epochs=30,
    lr=1e-2,
    device="cpu",
    seed=0
):
    rng = np.random.default_rng(seed)

    # 1) True embeddings + true user init states
    W_true = np_row_normalize(rng.normal(size=(n_items, d)))
    z0_true = np_row_normalize(rng.normal(size=(n_users, d)))

    # 2) Simulate interaction sequences
    seqs = simulate_sequences_from_true(
        W_true, z0_true, horizon,
        kappa=kappa, lam=lam, eta_pos=eta_pos, eta_neg=eta_neg,
        use_prox=use_prox,
        seed=seed + 123
    )

    # 3) Train/test split per user
    split = int(0.8 * horizon)
    train_seqs = [seq[:split] for seq in seqs]
    test_seqs  = [seq[split:] for seq in seqs]

    # 4) Learn embeddings from train sequences
    W_hat, z0_hat = fit_item_embeddings(
        train_seqs, n_items, d,
        kappa=kappa, lam=lam, eta_pos=eta_pos, eta_neg=eta_neg,
        use_prox=use_prox,
        learn_user_init=True,
        epochs=epochs,
        lr=lr,
        device=device,
        seed=seed
    )

    # 5) Recovery metrics
    m = embedding_metrics(W_true, W_hat, topk=5)

    # 6) Held-out predictive check (optional but useful)
    bce_true = heldout_bce(test_seqs, W_true, z0_true,
                           kappa=kappa, lam=lam, eta_pos=eta_pos, eta_neg=eta_neg, use_prox=use_prox)
    bce_hat  = heldout_bce(test_seqs, W_hat,  z0_hat,
                           kappa=kappa, lam=lam, eta_pos=eta_pos, eta_neg=eta_neg, use_prox=use_prox)

    return {
        "W_true": W_true, "z0_true": z0_true,
        "W_hat": W_hat, "z0_hat": z0_hat,
        "metrics": m,
        "heldout_bce_true": bce_true,
        "heldout_bce_hat": bce_hat
    }

if __name__ == "__main__":
    out = run_full_pipeline(
        n_items=60,
        n_users=40,
        d=8,
        horizon=200,
        kappa=5.0,
        lam=1.0,
        eta_pos=0.5,
        eta_neg=1.0,
        use_prox=True,
        epochs=30,
        lr=1e-2,
        device="cpu",
        seed=0
    )

    print("\nEmbedding recovery metrics:")
    for k, v in out["metrics"].items():
        print(f"{k:28s}: {v}")

    print("\nHeld-out response BCE (lower is better):")
    print("true W,true z0:", out["heldout_bce_true"])
    print("learned W,learned z0:", out["heldout_bce_hat"])

epoch 001 | mean loss/user = 87.6229
epoch 002 | mean loss/user = 47.7209
epoch 003 | mean loss/user = 40.8028
epoch 004 | mean loss/user = 38.5643
epoch 005 | mean loss/user = 37.2350
epoch 006 | mean loss/user = 36.2869
epoch 007 | mean loss/user = 35.6538
epoch 008 | mean loss/user = 34.8504
epoch 009 | mean loss/user = 34.4579
epoch 010 | mean loss/user = 34.0647
epoch 011 | mean loss/user = 33.5902
epoch 012 | mean loss/user = 33.1950
epoch 013 | mean loss/user = 32.8853
epoch 014 | mean loss/user = 32.6250
epoch 015 | mean loss/user = 32.3195
epoch 016 | mean loss/user = 32.1105
epoch 017 | mean loss/user = 31.9420
epoch 018 | mean loss/user = 31.7176
epoch 019 | mean loss/user = 31.5788
epoch 020 | mean loss/user = 31.5546
epoch 021 | mean loss/user = 31.3548
epoch 022 | mean loss/user = 31.2239
epoch 023 | mean loss/user = 31.0787
epoch 024 | mean loss/user = 31.0391
epoch 025 | mean loss/user = 30.9125
epoch 026 | mean loss/user = 30.8898
epoch 027 | mean loss/user = 30.8741
e